# 08 Evaluation

This notebook consolidates model prediction outputs into discrimination, calibration, confusion-matrix, and clinical-utility artifacts. It is designed to support both baseline and multimodal models through a common prediction-table format.

## Evaluation scope

- AUROC and AUPRC
- Precision, recall, F1, sensitivity, specificity
- Brier score and calibration tables
- ROC and PR curve points
- Lead-time summaries by horizon
- Reusable CSV artifacts for figure generation

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn

In [3]:
import pandas as pd

from src.evaluation.artifacts import build_lead_time_table, summarize_predictions
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [4]:
config['evaluation']

{'default_threshold': 0.5, 'calibration_bins': 10}

## Collect prediction tables

By default this notebook evaluates baseline prediction artifacts from Notebook 06. Later, multimodal prediction tables can be dropped into the same directory pattern and evaluated with the same logic.

In [5]:
baseline_dir = paths['processed_data_dir'] / '06_baseline_models'
multimodal_dir = paths['processed_data_dir'] / '07_multimodal_models'
prediction_files = sorted(baseline_dir.glob('*_test_predictions.csv')) + sorted(multimodal_dir.glob('*_test_predictions.csv'))
prediction_files = sorted(prediction_files)
assert prediction_files, 'No prediction files found. Run 06_baseline_models or 07_multimodal_models first.'
prediction_files[:10]

[PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_12h_logistic_regression_test_predictions.csv'),
 PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_12h_random_forest_test_predictions.csv'),
 PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_12h_xgboost_test_predictions.csv'),
 PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_24h_logistic_regression_test_predictions.csv'),
 PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_24h_random_forest_test_predictions.csv'),
 PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_baseline_models/horizon_24h_xgboost_test_predictions.csv'),
 PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/06_ba

## Evaluate each prediction file

In [6]:
metric_rows = []
artifact_bundle = {}

for path in prediction_files:
    predictions_df = pd.read_csv(path)
    model_name = predictions_df['model_name'].iloc[0] if 'model_name' in predictions_df.columns else path.stem
    dataset_name = predictions_df['dataset_name'].iloc[0] if 'dataset_name' in predictions_df.columns else 'unknown_dataset'
    horizon_hours = int(dataset_name.replace('horizon_', '').replace('h', '')) if dataset_name.startswith('horizon_') else -1

    metrics, curves = summarize_predictions(
        predictions_df=predictions_df,
        threshold=config['evaluation']['default_threshold'],
        calibration_bins=config['evaluation']['calibration_bins'],
    )
    metric_rows.append({
        'dataset_name': dataset_name,
        'model_name': model_name,
        'split': 'test',
        **metrics,
    })

    lead_time_df = build_lead_time_table(predictions_df, horizon_hours=horizon_hours)
    artifact_bundle[f'{dataset_name}_{model_name}_roc_curve'] = curves['roc_curve']
    artifact_bundle[f'{dataset_name}_{model_name}_pr_curve'] = curves['pr_curve']
    artifact_bundle[f'{dataset_name}_{model_name}_calibration'] = curves['calibration']
    artifact_bundle[f'{dataset_name}_{model_name}_lead_time'] = lead_time_df

evaluation_df = pd.DataFrame(metric_rows).sort_values(['dataset_name', 'model_name']).reset_index(drop=True)
evaluation_df

,dataset_name,model_name,split,auroc,auprc,precision,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn
0,horizon_12h,cross_modal_attention,test,0.618505,0.064832,0.000000,0.000000,0.000000,0.056640,1.000000,0.000000,0,0,5288,239
1,horizon_12h,early_fusion,test,0.783415,0.200648,0.310345,0.188285,0.234375,0.045689,0.981089,0.188285,45,100,5188,194
2,horizon_12h,gated_fusion,test,0.780048,0.176338,0.277778,0.062762,0.102389,0.040283,0.992625,0.062762,15,39,5249,224
3,horizon_12h,late_fusion,test,0.788869,0.219549,0.318471,0.209205,0.252525,0.046045,0.979766,0.209205,50,107,5181,189
4,horizon_12h,logistic_regression,test,0.853810,0.265201,0.152678,0.799163,0.256376,0.151014,0.799546,0.799163,191,1060,4228,48
5,horizon_12h,random_forest,test,0.998746,0.980104,0.983051,0.970711,0.976842,0.006747,0.999244,0.970711,232,4,5284,7
6,horizon_12h,xgboost,test,0.998611,0.961338,0.931174,0.962343,0.946502,0.004433,0.996785,0.962343,230,17,5271,9
7,horizon_24h,cross_modal_attention,test,0.602059,0.068442,0.000000,0.000000,0.000000,0.050124,1.000000,0.000000,0,0,3843,175
8,horizon_24h,early_fusion,test,0.754500,0.175086,0.364865,0.154286,0.216867,0.043443,0.987770,0.154286,27,47,3796,148
9,horizon_24h,gated_fusion,test,0.735367,0.158776,0.268293,0.125714,0.171206,0.046246,0.984387,0.125714,22,60,3783,153


## Inspect strongest test results

In [7]:
evaluation_df.sort_values('auprc', ascending=False).head(12) if not evaluation_df.empty else evaluation_df

,dataset_name,model_name,split,auroc,auprc,precision,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn
5,horizon_12h,random_forest,test,0.998746,0.980104,0.983051,0.970711,0.976842,0.006747,0.999244,0.970711,232,4,5284,7
19,horizon_6h,random_forest,test,0.997992,0.976293,0.982993,0.966555,0.974705,0.007575,0.999096,0.966555,289,5,5523,10
12,horizon_24h,random_forest,test,0.998785,0.975328,0.982955,0.988571,0.985755,0.005878,0.999219,0.988571,173,3,3840,2
20,horizon_6h,xgboost,test,0.998947,0.973890,0.939297,0.983278,0.960784,0.003763,0.996563,0.983278,294,19,5509,5
13,horizon_24h,xgboost,test,0.998962,0.965150,0.929348,0.977143,0.952646,0.003374,0.996617,0.977143,171,13,3830,4
6,horizon_12h,xgboost,test,0.998611,0.961338,0.931174,0.962343,0.946502,0.004433,0.996785,0.962343,230,17,5271,9
11,horizon_24h,logistic_regression,test,0.897076,0.339245,0.179975,0.811429,0.294606,0.116882,0.831642,0.811429,142,647,3196,33
4,horizon_12h,logistic_regression,test,0.853810,0.265201,0.152678,0.799163,0.256376,0.151014,0.799546,0.799163,191,1060,4228,48
16,horizon_6h,gated_fusion,test,0.808292,0.264337,0.352941,0.260870,0.300000,0.052045,0.974132,0.260870,78,143,5385,221
15,horizon_6h,early_fusion,test,0.799145,0.258055,0.355556,0.267559,0.305344,0.051462,0.973770,0.267559,80,145,5383,219


## Save evaluation artifacts

In [8]:
output_dir = paths['processed_data_dir'] / '08_evaluation'
artifact_bundle['evaluation_summary'] = evaluation_df
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

{'horizon_12h_logistic_regression_roc_curve': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/08_evaluation/horizon_12h_logistic_regression_roc_curve.csv',
 'horizon_12h_logistic_regression_pr_curve': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/08_evaluation/horizon_12h_logistic_regression_pr_curve.csv',
 'horizon_12h_logistic_regression_calibration': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/08_evaluation/horizon_12h_logistic_regression_calibration.csv',
 'horizon_12h_logistic_regression_lead_time': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/08_evaluation/horizon_12h_logistic_regression_lead_time.csv',
 'horizon_12h_random_forest_roc_curve': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed/08_evaluation/horizon_12h_random_forest_roc_curve.csv',
 'horizon_12h_random_forest_pr_curve': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/processed

In [9]:
manifest_path = paths['manifests_dir'] / '08_evaluation_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='08_evaluation',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'prediction_files_evaluated': [str(path) for path in prediction_files],
        'num_models_evaluated': int(evaluation_df['model_name'].nunique()) if not evaluation_df.empty else 0,
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main/results/manifests/08_evaluation_manifest.json')

## Review checklist

Before ablation analysis, review:

- Which model wins on AUROC versus AUPRC?
- Are calibration tables noticeably misaligned?
- Does performance degrade as horizon length increases?
- Are sensitivity and specificity balanced enough for clinical use?
- Which prediction files should be carried into the paper figures?

## Next notebook

`09_ablation_experiments.ipynb` will compare modality subsets, temporal feature subsets, and fusion strategies to strengthen the research contribution.